# Phase 3 — Model A: Traditional ML (TF-IDF, Colab-Ready)

**Models trained:**
1. Logistic Regression (TF-IDF + Lexical features)
2. SVM / LinearSVC (TF-IDF + Lexical, decision_function for EM)
3. Naive Bayes (Question Type Classification)
4. XGBoost (Lexical features, bonus)

**Question Generation:**
- Template-based Wh-word generation
- ML Ranker (Random Forest) to select best candidate

**Evaluation:**
- Answer Verification: BLEU, ROUGE-1, ROUGE-2, ROUGE-L, METEOR (predicted vs gold answer text)
- Question Generation: BLEU, ROUGE-1, ROUGE-2, ROUGE-L, METEOR (generated vs reference question)
- Naive Bayes: per-class F1 / classification report

**Expected time on Colab T4: ~60-90 minutes**

## 1. Mount Drive and Setup

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/My Drive/race_rc_proj'

if not os.path.exists(PROJECT_PATH):
    raise FileNotFoundError(f'Project not found at: {PROJECT_PATH}')

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

## 2. Imports and Dependencies

In [ ]:
import os, re, json, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy import sparse
from scipy.sparse import hstack

import joblib
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer as QTypeVec, TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.exceptions import ConvergenceWarning

# NLP evaluation metrics
!pip install rouge-score nltk -q
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore')
np.random.seed(42)

print('All imports done.')

## 3. Load Preprocessed Data

In [ ]:
DATA_DIR  = 'data/processed'
MODEL_DIR = 'models/model_a/traditional'
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

print('Loading preprocessed feature files...')

X_train_tfidf = sparse.load_npz(f'{DATA_DIR}/X_train_tfidf.npz')
X_dev_tfidf   = sparse.load_npz(f'{DATA_DIR}/X_dev_tfidf.npz')
X_test_tfidf  = sparse.load_npz(f'{DATA_DIR}/X_test_tfidf.npz')

X_train_ohe = sparse.load_npz(f'{DATA_DIR}/X_train_ohe.npz')
X_dev_ohe   = sparse.load_npz(f'{DATA_DIR}/X_dev_ohe.npz')
X_test_ohe  = sparse.load_npz(f'{DATA_DIR}/X_test_ohe.npz')

X_train_lex = np.load(f'{DATA_DIR}/X_train_lexical.npy')
X_dev_lex   = np.load(f'{DATA_DIR}/X_dev_lexical.npy')
X_test_lex  = np.load(f'{DATA_DIR}/X_test_lexical.npy')

y_train = np.load(f'{DATA_DIR}/y_train.npy')
y_dev   = np.load(f'{DATA_DIR}/y_dev.npy')
y_test  = np.load(f'{DATA_DIR}/y_test.npy')

row_ids_train = np.load(f'{DATA_DIR}/row_ids_train.npy')
row_ids_dev   = np.load(f'{DATA_DIR}/row_ids_dev.npy')
row_ids_test  = np.load(f'{DATA_DIR}/row_ids_test.npy')

tfidf_vectorizer = joblib.load(f'{MODEL_DIR}/tfidf_vectorizer.pkl')

print(f'Train TF-IDF:  {X_train_tfidf.shape}')
print(f'Train Lexical: {X_train_lex.shape}')
print(f'Train labels:  {y_train.shape}')
print(f'Train positive ratio: {y_train.mean():.4f}  (should be ~0.25)')
print(f'Unique train questions: {len(np.unique(row_ids_train)):,}')
print(f'Unique dev   questions: {len(np.unique(row_ids_dev)):,}')
print(f'Unique test  questions: {len(np.unique(row_ids_test)):,}')

### 3a. Sanity Checks

In [ ]:
counts = pd.Series(row_ids_train).value_counts()
assert (counts == 4).all(), f'row_ids broken: {counts.value_counts().to_dict()}'

df_check = pd.DataFrame({'row_id': row_ids_train, 'y': y_train})
pos_per_q = df_check.groupby('row_id')['y'].sum()
assert (pos_per_q == 1).all(), f'Label structure broken: {pos_per_q.value_counts().to_dict()}'

print('Sanity checks passed — row_ids and labels correctly structured.')
del df_check

## 4. Prepare Combined Feature Matrices

TF-IDF features are L2-normalised floats. Lexical features are scaled with
`StandardScaler` (zero-centred, compatible with TF-IDF floats) then hstacked.

In [ ]:
scaler = StandardScaler()
X_train_lex_scaled = scaler.fit_transform(X_train_lex)
X_dev_lex_scaled   = scaler.transform(X_dev_lex)
X_test_lex_scaled  = scaler.transform(X_test_lex)

X_train_combined = hstack([X_train_tfidf, sparse.csr_matrix(X_train_lex_scaled)]).tocsr()
X_dev_combined   = hstack([X_dev_tfidf,   sparse.csr_matrix(X_dev_lex_scaled)]).tocsr()
X_test_combined  = hstack([X_test_tfidf,  sparse.csr_matrix(X_test_lex_scaled)]).tocsr()

joblib.dump(scaler, f'{MODEL_DIR}/scaler_lexical.pkl')

print(f'Combined train shape: {X_train_combined.shape}')
print(f'  TF-IDF dims:  {X_train_tfidf.shape[1]:,}')
print(f'  Lexical dims: {X_train_lex.shape[1]}')
print(f'  Format: {X_train_combined.format}  (must be csr)')
print('Scaler saved.')

## 5. Evaluation Functions

In [ ]:
# ── Traditional metric helpers ────────────────────────────────────────────

def compute_exact_match(probs, row_ids, y_true):
    """Pick highest-scoring option per question; EM = fraction correct."""
    df = pd.DataFrame({'row_id': row_ids, 'prob': probs, 'correct': y_true})
    def check_q(group):
        best_idx = group['prob'].idxmax()
        return int(group.loc[best_idx, 'correct']) == 1
    return df.groupby('row_id').apply(check_q).mean()


def evaluate_model(model, X, y_true, row_ids, model_name, use_decision_fn=False):
    """Binary metrics + Exact Match."""
    y_pred = model.predict(X)
    scores = model.decision_function(X) if use_decision_fn else model.predict_proba(X)[:, 1]
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro')
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    em   = compute_exact_match(scores, row_ids, y_true)
    cm   = confusion_matrix(y_true, y_pred)
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    print(f'{"="*60}')
    print(f'  Accuracy:    {acc:.4f}')
    print(f'  Macro F1:    {f1:.4f}')
    print(f'  Precision:   {prec:.4f}')
    print(f'  Recall:      {rec:.4f}')
    print(f'  Exact Match: {em:.4f}  <- main metric')
    print(f'\n  Confusion Matrix:')
    print(f'    TN={cm[0,0]:>7,}  FP={cm[0,1]:>7,}')
    print(f'    FN={cm[1,0]:>7,}  TP={cm[1,1]:>7,}')
    return {'model_name': model_name, 'accuracy': acc, 'macro_f1': f1,
            'precision': prec, 'recall': rec, 'exact_match': em,
            'confusion_matrix': cm.tolist()}


def evaluate_svm_final(svc, cal_model, X, y_true, row_ids, model_name):
    """SVM eval: decision_function for EM, calibrated predict() for binary."""
    scores = svc.decision_function(X)
    em     = compute_exact_match(scores, row_ids, y_true)
    y_pred = cal_model.predict(X) if cal_model is not None else svc.predict(X)
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro')
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    cm   = confusion_matrix(y_true, y_pred)
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    print(f'{"="*60}')
    print(f'  Accuracy:    {acc:.4f}')
    print(f'  Macro F1:    {f1:.4f}')
    print(f'  Precision:   {prec:.4f}')
    print(f'  Recall:      {rec:.4f}')
    print(f'  Exact Match: {em:.4f}  <- main metric  [decision_function]')
    print(f'\n  Confusion Matrix:')
    print(f'    TN={cm[0,0]:>7,}  FP={cm[0,1]:>7,}')
    print(f'    FN={cm[1,0]:>7,}  TP={cm[1,1]:>7,}')
    return {'model_name': model_name, 'accuracy': acc, 'macro_f1': f1,
            'precision': prec, 'recall': rec, 'exact_match': em,
            'confusion_matrix': cm.tolist()}


def plot_confusion_matrix(model, X, y_true, title, save_path=None):
    y_pred = model.predict(X)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=['Wrong (0)', 'Correct (1)'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# ── NLP metric helper ──────────────────────────────────────────────────────

def evaluate_nlp_metrics(predictions, references):
    """BLEU, ROUGE-1/2/L, METEOR between two lists of strings."""
    smoother = SmoothingFunction().method1
    rscorer  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    bleu_s, meteor_s, r1_s, r2_s, rL_s = [], [], [], [], []
    for pred, ref in zip(predictions, references):
        p_tok, r_tok = pred.split(), ref.split()
        bleu_s.append(sentence_bleu([r_tok], p_tok, smoothing_function=smoother))
        meteor_s.append(meteor_score([r_tok], p_tok))
        s = rscorer.score(ref, pred)
        r1_s.append(s['rouge1'].fmeasure)
        r2_s.append(s['rouge2'].fmeasure)
        rL_s.append(s['rougeL'].fmeasure)
    return {
        'BLEU':    round(np.mean(bleu_s),   4),
        'ROUGE-1': round(np.mean(r1_s),     4),
        'ROUGE-2': round(np.mean(r2_s),     4),
        'ROUGE-L': round(np.mean(rL_s),     4),
        'METEOR':  round(np.mean(meteor_s), 4),
    }


# ── Answer text NLP helper ─────────────────────────────────────────────────

def letter_to_idx(letter):
    return {'A': 0, 'B': 1, 'C': 2, 'D': 3}.get(str(letter).strip().upper(), 0)

def get_pred_gold_texts(scores, row_ids, dev_raw_df):
    """
    Given ranking scores (one per option-row) and a raw dev DataFrame,
    return (predicted_answer_texts, gold_answer_texts).
    """
    id_col  = 'id' if 'id' in dev_raw_df.columns else dev_raw_df.columns[0]
    id_to_row = dev_raw_df.set_index(id_col).to_dict('index')
    df_tmp  = pd.DataFrame({'row_id': row_ids, 'score': scores})
    pred_texts, gold_texts = [], []
    for qid, group in df_tmp.groupby('row_id'):
        if qid not in id_to_row:
            continue
        row      = id_to_row[qid]
        options  = [str(row.get('A','')), str(row.get('B','')),
                    str(row.get('C','')), str(row.get('D',''))]
        pred_idx = int(group['score'].values.argmax())
        gold_idx = letter_to_idx(row.get('answer', 'A'))
        pred_texts.append(options[pred_idx])
        gold_texts.append(options[gold_idx])
    return pred_texts, gold_texts


print('All evaluation functions defined.')

## 6. Load Raw CSV Data (needed for NLP metrics and question generation)

In [ ]:
train_raw = pd.read_csv('data/raw/train.csv')

dev_path = 'data/raw/dev.csv' if os.path.exists('data/raw/dev.csv') else 'data/raw/val.csv'
assert os.path.exists(dev_path), 'Neither dev.csv nor val.csv found in data/raw/'
dev_raw = pd.read_csv(dev_path)

print(f'train_raw shape: {train_raw.shape}')
print(f'dev_raw   shape: {dev_raw.shape}')
print(f'Columns: {list(train_raw.columns)}')

## 7. Logistic Regression

**Why `liblinear`:** coordinate descent, best for high-dimensional sparse text
(281k rows × 10k features). `lbfgs`/`saga` hit `max_iter` on this data.

**Strategy:** sweep C on TF-IDF only → train final on TF-IDF + Lexical combined
→ pick whichever wins on dev Exact Match.

In [ ]:
print('='*60)
print('LR — C SWEEP (liblinear, TF-IDF only)')
print('~6-10 minutes')
print('='*60)

lr_sweep_results = []

for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    t0 = time.time()
    print(f'  C={C}...', end=' ', flush=True)
    m = LogisticRegression(C=C, solver='liblinear', max_iter=2000,
                           class_weight='balanced', random_state=42)
    m.fit(X_train_tfidf, y_train)
    proba = m.predict_proba(X_dev_tfidf)[:, 1]
    em    = compute_exact_match(proba, row_ids_dev, y_dev)
    acc   = accuracy_score(y_dev, m.predict(X_dev_tfidf))
    elapsed = time.time() - t0
    conv  = 'OK' if m.n_iter_[0] < 2000 else 'NOT CONVERGED'
    print(f'Acc={acc:.4f}  EM={em:.4f}  {conv}({m.n_iter_[0]} iters)  ({elapsed:.0f}s)')
    lr_sweep_results.append({'C': C, 'Accuracy': acc, 'Exact Match': em,
                              'n_iter': m.n_iter_[0]})

lr_sweep_df = pd.DataFrame(lr_sweep_results)
best_C_lr   = lr_sweep_df.loc[lr_sweep_df['Exact Match'].idxmax(), 'C']
print(f'\nC Sweep Results:')
print(lr_sweep_df.to_string(index=False))
print(f'\nBest C: {best_C_lr}  EM={lr_sweep_df["Exact Match"].max():.4f}')

In [ ]:
# Train final LR — TF-IDF only
print(f'Training LR (C={best_C_lr}) on TF-IDF features...')
lr_tfidf = LogisticRegression(C=best_C_lr, solver='liblinear', max_iter=2000,
                               class_weight='balanced', random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
print(f'Converged in {lr_tfidf.n_iter_[0]} iterations')

lr_tfidf_dev  = evaluate_model(lr_tfidf, X_dev_tfidf,  y_dev,  row_ids_dev,  'LR TF-IDF (Dev)')
lr_tfidf_test = evaluate_model(lr_tfidf, X_test_tfidf, y_test, row_ids_test, 'LR TF-IDF (Test)')

plot_confusion_matrix(lr_tfidf, X_dev_tfidf, y_dev,
                      'LR (TF-IDF) — Dev Set',
                      save_path=f'{MODEL_DIR}/lr_tfidf_confusion_matrix.png')

In [ ]:
# Train LR — TF-IDF + Lexical combined
print(f'Training LR (C={best_C_lr}) on TF-IDF + Lexical combined features...')
lr_combined = LogisticRegression(C=best_C_lr, solver='liblinear', max_iter=2000,
                                  class_weight='balanced', random_state=42)
lr_combined.fit(X_train_combined, y_train)
print(f'Converged in {lr_combined.n_iter_[0]} iterations')

lr_comb_dev  = evaluate_model(lr_combined, X_dev_combined,  y_dev,  row_ids_dev,  'LR Combined (Dev)')
lr_comb_test = evaluate_model(lr_combined, X_test_combined, y_test, row_ids_test, 'LR Combined (Test)')

# Pick best variant by dev EM
if lr_comb_dev['exact_match'] >= lr_tfidf_dev['exact_match']:
    lr_model       = lr_combined
    lr_dev_metrics = lr_comb_dev
    lr_test_metrics= lr_comb_test
    lr_X_dev       = X_dev_combined
    lr_X_test      = X_test_combined
    print('\nTF-IDF + Lexical won → using LR Combined as final model.')
else:
    lr_model       = lr_tfidf
    lr_dev_metrics = lr_tfidf_dev
    lr_test_metrics= lr_tfidf_test
    lr_X_dev       = X_dev_tfidf
    lr_X_test      = X_test_tfidf
    print('\nTF-IDF only won → using LR TF-IDF as final model.')

plot_confusion_matrix(lr_model, lr_X_dev, y_dev,
                      'LR Final — Dev Set',
                      save_path=f'{MODEL_DIR}/lr_final_confusion_matrix.png')

joblib.dump(lr_model, f'{MODEL_DIR}/logistic_regression.pkl')
print(f'Saved: {MODEL_DIR}/logistic_regression.pkl')

## 8. SVM (LinearSVC)

**Strategy:** 85/15 prefit calibration split → C sweep with decision_function EM
→ final model saved as both raw SVC (for EM ranking) and calibrated wrapper.

In [ ]:
# 85/15 split for prefit calibration — prevents CV collapse
(X_svm_fit, X_svm_cal,
 y_svm_fit, y_svm_cal,
 ids_svm_fit, ids_svm_cal) = train_test_split(
    X_train_combined, y_train, row_ids_train,
    test_size=0.15, stratify=y_train, random_state=42
)
X_svm_fit = X_svm_fit.tocsr()
X_svm_cal = X_svm_cal.tocsr()

print(f'SVC training set:   {X_svm_fit.shape[0]:,} rows')
print(f'Calibration set:    {X_svm_cal.shape[0]:,} rows')
print(f'Cal positive ratio: {y_svm_cal.mean():.3f}')

In [ ]:
print('='*60)
print('SVM — C SWEEP (decision_function EM, prefit calibration)')
print('~15-20 minutes')
print('='*60)

svm_sweep_results = []

for C in [0.001, 0.01, 0.1, 1.0]:
    t0 = time.time()
    print(f'\nC={C}...', end=' ', flush=True)

    svc = LinearSVC(C=C, max_iter=5000, loss='squared_hinge',
                    class_weight='balanced', random_state=42)
    svc.fit(X_svm_fit, y_svm_fit)

    df_scores = svc.decision_function(X_dev_combined)
    em        = compute_exact_match(df_scores, row_ids_dev, y_dev)

    cal = CalibratedClassifierCV(svc, cv='prefit', method='sigmoid')
    cal.fit(X_svm_cal, y_svm_cal)
    y_pred = cal.predict(X_dev_combined)
    acc    = accuracy_score(y_dev, y_pred)
    prec   = precision_score(y_dev, y_pred, zero_division=0)
    elapsed = time.time() - t0

    status = 'COLLAPSED' if prec < 0.01 else 'OK'
    print(f'Acc={acc:.4f}  EM={em:.4f}  Prec={prec:.4f}  ({elapsed:.0f}s) {status}')
    svm_sweep_results.append({'C': C, 'Accuracy': acc, 'Exact Match': em,
                               'Precision': prec, 'n_iter': svc.n_iter_,
                               'svc': svc, 'cal': cal})

valid = [r for r in svm_sweep_results if r['Precision'] > 0.01]
if valid:
    best_svm_row = max(valid, key=lambda x: x['Exact Match'])
else:
    best_svm_row = max(svm_sweep_results, key=lambda x: x['Exact Match'])

best_C_svm = best_svm_row['C']
svm_sweep_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('svc','cal')}
                               for r in svm_sweep_results])
print(f'\nSVM Sweep Results:')
print(svm_sweep_df.to_string(index=False))
print(f'\nBest C: {best_C_svm}  EM={best_svm_row["Exact Match"]:.4f}')

In [ ]:
print(f'Training FINAL SVM with C={best_C_svm}...')
t0 = time.time()

svc_final = LinearSVC(C=best_C_svm, max_iter=5000, loss='squared_hinge',
                      class_weight='balanced', random_state=42)
svc_final.fit(X_svm_fit, y_svm_fit)
print(f'SVC n_iter_: {svc_final.n_iter_}')

svm_model = CalibratedClassifierCV(svc_final, cv='prefit', method='sigmoid')
svm_model.fit(X_svm_cal, y_svm_cal)

# Collapse check
y_sample = svm_model.predict(X_dev_combined[:2000])
if y_sample.sum() == 0:
    print('Sigmoid collapsed — trying isotonic...')
    svm_iso = CalibratedClassifierCV(svc_final, cv='prefit', method='isotonic')
    svm_iso.fit(X_svm_cal, y_svm_cal)
    svm_model = svm_iso if svm_iso.predict(X_dev_combined[:2000]).sum() > 0 else None
    print('Isotonic used.' if svm_model else 'Both collapsed — EM still valid via decision_function.')
else:
    print(f'Calibration OK. Predicted positives sample: {y_sample.sum()}')

print(f'Total time: {(time.time()-t0)/60:.1f} min')

svm_dev_metrics  = evaluate_svm_final(svc_final, svm_model, X_dev_combined,  y_dev,  row_ids_dev,  'SVM (Dev)')
svm_test_metrics = evaluate_svm_final(svc_final, svm_model, X_test_combined, y_test, row_ids_test, 'SVM (Test)')

plot_confusion_matrix(svc_final, X_dev_combined, y_dev,
                      'SVM — Dev Set',
                      save_path=f'{MODEL_DIR}/svm_confusion_matrix.png')

joblib.dump(svc_final, f'{MODEL_DIR}/svm_linearsvc.pkl')
if svm_model:
    joblib.dump(svm_model, f'{MODEL_DIR}/svm_calibrated.pkl')
print(f'Saved: svm_linearsvc.pkl + svm_calibrated.pkl')

## 9. Naive Bayes — Question Type Classifier

In [ ]:
WH_WORDS = ['who', 'what', 'where', 'when', 'why', 'how', 'which']

def get_question_type(q):
    tokens = str(q).strip().lower().split()
    first  = tokens[0] if tokens else ''
    for wh in WH_WORDS:
        if first.startswith(wh):
            return wh
    return 'other'

train_raw['qtype'] = train_raw['question'].apply(get_question_type)
dev_raw['qtype']   = dev_raw['question'].apply(get_question_type)

print('Question type distribution (train):')
print(train_raw['qtype'].value_counts().to_string())

In [ ]:
le = LabelEncoder()
y_qt_train = le.fit_transform(train_raw['qtype'])
y_qt_dev   = le.transform(dev_raw['qtype'])

qt_vec = QTypeVec(binary=False, max_features=2000, min_df=2)
X_qt_train = qt_vec.fit_transform(train_raw['question'].fillna(''))
X_qt_dev   = qt_vec.transform(dev_raw['question'].fillna(''))

nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_qt_train, y_qt_train)

y_qt_pred = nb_model.predict(X_qt_dev)
print('Naive Bayes — Question Type Classification (Dev):')
print(classification_report(y_qt_dev, y_qt_pred, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_qt_dev, y_qt_pred,
    display_labels=le.classes_,
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title('Naive Bayes — Question Type (Dev)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/nb_qtype_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

joblib.dump(nb_model, f'{MODEL_DIR}/nb_qtype_model.pkl')
joblib.dump(qt_vec,   f'{MODEL_DIR}/nb_qtype_vectorizer.pkl')
joblib.dump(le,       f'{MODEL_DIR}/nb_qtype_label_encoder.pkl')
print('Saved: nb_qtype_model.pkl + vectorizer + label_encoder')

## 10. XGBoost — Lexical Features Only (Bonus)

In [ ]:
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print('XGBoost not installed. Run: !pip install xgboost')
    HAS_XGB = False

if HAS_XGB:
    print('='*60)
    print('XGBOOST — Lexical Features Only')
    print('~5-10 minutes')
    print('='*60)

    xgb = XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        eval_metric='logloss', scale_pos_weight=3,
        early_stopping_rounds=20, n_jobs=-1,
        random_state=42, verbosity=0
    )
    t0 = time.time()
    xgb.fit(X_train_lex, y_train, eval_set=[(X_dev_lex, y_dev)], verbose=50)
    print(f'\nXGBoost trained in {(time.time()-t0)/60:.1f} min')
    print(f'Best iteration: {xgb.best_iteration}')

    xgb_dev_metrics  = evaluate_model(xgb, X_dev_lex,  y_dev,  row_ids_dev,  'XGBoost Lexical (Dev)')
    xgb_test_metrics = evaluate_model(xgb, X_test_lex, y_test, row_ids_test, 'XGBoost Lexical (Test)')

    plot_confusion_matrix(xgb, X_dev_lex, y_dev,
                          'XGBoost (Lexical) — Dev Set',
                          save_path=f'{MODEL_DIR}/xgb_confusion_matrix.png')

    try:
        with open('data/processed/preprocessing_config.json') as f:
            config = json.load(f)
        feat_names = config.get('lexical_feature_names',
                                [f'feat_{i}' for i in range(X_train_lex.shape[1])])
    except Exception:
        feat_names = [f'feat_{i}' for i in range(X_train_lex.shape[1])]

    feat_df = pd.DataFrame({'Feature': feat_names,
                             'Importance': xgb.feature_importances_}).sort_values('Importance')
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue')
    ax.set_title('XGBoost Feature Importances (Lexical)', fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig(f'{MODEL_DIR}/xgb_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    joblib.dump(xgb, f'{MODEL_DIR}/xgb_model.pkl')
    print(f'Saved: {MODEL_DIR}/xgb_model.pkl')

## 11. NLP Evaluation Metrics — Answer Verification

Per instructor update, evaluation uses **BLEU, ROUGE, METEOR** instead of
accuracy/precision. For answer verification, we compare the **predicted answer
text** (option chosen by the model) against the **gold answer text**.

In [ ]:
print('Computing NLP metrics for answer verification models...')
print('(Comparing predicted answer text vs gold answer text)')
print()

# ── LR ──
lr_scores          = lr_model.predict_proba(lr_X_dev)[:, 1]
lr_pred_texts, gold_texts = get_pred_gold_texts(lr_scores, row_ids_dev, dev_raw)
lr_nlp             = evaluate_nlp_metrics(lr_pred_texts, gold_texts)
print('LR NLP metrics computed.')

# ── SVM ──
svm_scores         = svc_final.decision_function(X_dev_combined)
svm_pred_texts, _  = get_pred_gold_texts(svm_scores, row_ids_dev, dev_raw)
svm_nlp            = evaluate_nlp_metrics(svm_pred_texts, gold_texts)
print('SVM NLP metrics computed.')

# ── XGBoost ──
if HAS_XGB:
    xgb_scores        = xgb.predict_proba(X_dev_lex)[:, 1]
    xgb_pred_texts, _ = get_pred_gold_texts(xgb_scores, row_ids_dev, dev_raw)
    xgb_nlp           = evaluate_nlp_metrics(xgb_pred_texts, gold_texts)
    print('XGBoost NLP metrics computed.')

# ── Table ──
nlp_rows = {'LR': lr_nlp, 'SVM': svm_nlp}
if HAS_XGB:
    nlp_rows['XGBoost'] = xgb_nlp

nlp_df = pd.DataFrame(nlp_rows).T

print('\n' + '='*65)
print('  NLP METRICS — ANSWER VERIFICATION (Predicted vs Gold Answer Text)')
print('='*65)
print(nlp_df.to_string())
print('='*65)

## 12. Template-Based Question Generation

Generates candidate questions from anchor sentences using Wh-word templates,
then an ML Ranker (Random Forest) scores and selects the best candidate.

In [ ]:
COMMON_CAPS = {
    'The','In','It','At','On','By','An','A','This','That','These','Those',
    'He','She','They','We','I','His','Her','Their','Its','Our','Was','Were',
    'Has','Have','Had','Is','Are','Am','But','And','Or','So','Yet','For'
}

def detect_name(tokens, idx):
    w = tokens[idx]
    return (len(w) > 1 and w[0].isupper() and idx != 0
            and w not in COMMON_CAPS and w.isalpha())

def generate_candidates_from_sentence(sentence):
    """Apply Wh-word templates to a sentence; return list of candidate questions."""
    tokens     = sentence.split()
    candidates = []

    # When — year detected
    for t in tokens:
        if re.fullmatch(r'\d{4}', t):
            rest = sentence.replace(t, '___')
            candidates.append(f'When {rest}?')
            break

    # Where — word after 'in' / 'at'
    for i, t in enumerate(tokens):
        if t.lower() in ('in', 'at') and i + 1 < len(tokens):
            candidates.append(f'Where {" ".join(tokens[i+1:])}?')
            break

    # Why — reason clause
    for i, t in enumerate(tokens):
        if t.lower() in ('because', 'since') and i + 1 < len(tokens):
            candidates.append(f'Why {" ".join(tokens[i+1:])}?')
            break

    # Who — proper name
    for i, t in enumerate(tokens):
        if detect_name(tokens, i):
            rest = ' '.join(tok for j, tok in enumerate(tokens) if j != i)
            candidates.append(f'Who {rest}?')
            break

    # What — default
    if not candidates:
        short = ' '.join(tokens[:8])
        candidates.append(f'What is meant by "{short}"?')

    return candidates

def select_top_sentences(article, answer_text, top_n=3):
    """Return top_n sentences ranked by word overlap with answer."""
    sentences    = [s.strip() for s in re.split(r'\.\s+', article) if len(s.strip()) > 20]
    answer_words = set(answer_text.lower().split())
    scored = [(len(set(s.lower().split()) & answer_words), i, s)
              for i, s in enumerate(sentences)]
    scored.sort(reverse=True)
    return scored[:top_n], len(sentences)

print('Question generation helpers defined.')

### 12a. Train ML Ranker for Question Candidates

In [ ]:
def ranker_features(question, anchor, anchor_pos, article_len):
    """5-feature vector for a (question, anchor) pair."""
    q_tok    = question.split()
    a_tok    = anchor.split()
    wh_words = {'who','what','where','when','why','how','which'}
    overlap  = len(set(t.lower() for t in q_tok) & set(t.lower() for t in a_tok))
    has_wh   = int(any(t.lower() in wh_words for t in q_tok))
    return [
        len(q_tok),                          # question length
        has_wh,                              # contains Wh-word (0/1)
        overlap,                             # overlap with anchor
        anchor_pos / max(article_len, 1),    # relative anchor position
        len(a_tok),                          # anchor sentence length
    ]

def label_candidate(gen_q, ref_q, threshold=2):
    """1 if word overlap with reference question >= threshold."""
    return int(len(set(gen_q.lower().split()) & set(ref_q.lower().split())) >= threshold)

print('Building ranker training data from 3,000 train samples...')
sample_train = train_raw.dropna(subset=['article','answer','question']).head(3000)

ranker_X, ranker_y = [], []
for _, row in sample_train.iterrows():
    article    = str(row['article'])
    ans_col    = {'A':'A','B':'B','C':'C','D':'D'}.get(str(row['answer']).strip().upper(), 'A')
    answer_txt = str(row.get(ans_col, ''))
    ref_q      = str(row['question'])

    top_sents, art_len = select_top_sentences(article, answer_txt, top_n=3)
    for _, pos, anchor in top_sents:
        for cand in generate_candidates_from_sentence(anchor):
            ranker_X.append(ranker_features(cand, anchor, pos, art_len))
            ranker_y.append(label_candidate(cand, ref_q))

ranker_X = np.array(ranker_X)
ranker_y = np.array(ranker_y)
print(f'Ranker dataset: {len(ranker_X):,} candidates  |  positive rate: {ranker_y.mean():.2f}')

qg_ranker = RandomForestClassifier(n_estimators=100, max_depth=8,
                                    class_weight='balanced', random_state=42, n_jobs=-1)
qg_ranker.fit(ranker_X, ranker_y)
print('ML Ranker trained.')

joblib.dump(qg_ranker, f'{MODEL_DIR}/qg_ranker.pkl')
print(f'Saved: {MODEL_DIR}/qg_ranker.pkl')

### 12b. Generate Questions and Evaluate with BLEU / ROUGE / METEOR

In [ ]:
def generate_question_ranked(article, answer_text):
    """Generate best question using template + ML ranker."""
    top_sents, art_len = select_top_sentences(article, answer_text, top_n=3)
    all_candidates = []
    for _, pos, anchor in top_sents:
        for cand in generate_candidates_from_sentence(anchor):
            feats = ranker_features(cand, anchor, pos, art_len)
            score = qg_ranker.predict_proba([feats])[0][1]
            all_candidates.append((score, cand, anchor))
    if not all_candidates:
        return 'What is the main idea of the passage?', ''
    all_candidates.sort(reverse=True)
    return all_candidates[0][1], all_candidates[0][2]   # best question, anchor


print('Evaluating ML-ranked question generation on 500 dev samples...')
sample_dev = dev_raw.dropna(subset=['article','answer','question']).head(500).copy()

gen_questions_ranked, ref_questions = [], []
gen_questions_template = []   # baseline: template-only (no ranker, just first candidate)

for _, row in sample_dev.iterrows():
    ans_col    = {'A':'A','B':'B','C':'C','D':'D'}.get(str(row['answer']).strip().upper(), 'A')
    answer_txt = str(row.get(ans_col, ''))
    ref_q      = str(row['question'])

    # ML-ranked generation
    gen_q_ranked, _ = generate_question_ranked(str(row['article']), answer_txt)
    gen_questions_ranked.append(gen_q_ranked)

    # Template-only baseline (first candidate from best anchor sentence)
    top_sents, _ = select_top_sentences(str(row['article']), answer_txt, top_n=1)
    cands = generate_candidates_from_sentence(top_sents[0][2]) if top_sents else ['What is the main idea?']
    gen_questions_template.append(cands[0])

    ref_questions.append(ref_q)

qg_ranked_metrics   = evaluate_nlp_metrics(gen_questions_ranked,   ref_questions)
qg_template_metrics = evaluate_nlp_metrics(gen_questions_template, ref_questions)

print('\n' + '='*60)
print('  QUESTION GENERATION — NLP METRICS (Dev, 500 samples)')
print('='*60)
print(f'  {"Metric":<12} {"Template-Only":>15} {"ML Ranker":>12}')
print(f'  {"-"*40}')
for k in qg_ranked_metrics:
    print(f'  {k:<12} {qg_template_metrics[k]:>15.4f} {qg_ranked_metrics[k]:>12.4f}')
print('='*60)

## 13. Final Comparison Tables

In [ ]:
# ── Traditional metrics table ─────────────────────────────────────────────
all_metrics = {
    'LR (Dev)':   lr_dev_metrics,
    'LR (Test)':  lr_test_metrics,
    'SVM (Dev)':  svm_dev_metrics,
    'SVM (Test)': svm_test_metrics,
}
if HAS_XGB:
    all_metrics['XGBoost (Dev)']  = xgb_dev_metrics
    all_metrics['XGBoost (Test)'] = xgb_test_metrics

rows = []
for name, m in all_metrics.items():
    rows.append({
        'Model/Split': name,
        'Accuracy':    round(m['accuracy'],    4),
        'Macro F1':    round(m['macro_f1'],    4),
        'Exact Match': round(m['exact_match'], 4),
    })
results_df = pd.DataFrame(rows).set_index('Model/Split')

print('\n' + '='*70)
print('  PHASE 3 — COMPLETE EVALUATION SUMMARY')
print('='*70)

print('\n[1] ANSWER VERIFICATION — Traditional Metrics (EM = primary metric)')
print(results_df.to_string())

print('\n[2] ANSWER VERIFICATION — NLP Metrics (Predicted vs Gold Answer Text)')
print(nlp_df.to_string())

print('\n[3] QUESTION GENERATION — NLP Metrics')
print(f'  {"Metric":<12} {"Template-Only":>15} {"ML Ranker":>12}')
print(f'  {"-"*40}')
for k in qg_ranked_metrics:
    print(f'  {k:<12} {qg_template_metrics[k]:>15.4f} {qg_ranked_metrics[k]:>12.4f}')

print('\n[4] NAIVE BAYES — Question Type Classification')
print('  (See classification_report output above in Section 9)')

print('\n' + '='*70)

dev_rows = {k: v for k, v in all_metrics.items() if '(Dev)' in k}
best     = max(dev_rows, key=lambda k: dev_rows[k]['exact_match'])
print(f'  Best model (Dev EM): {best} — EM={dev_rows[best]["exact_match"]:.4f}')
print('='*70)

## 14. Save All Results

In [ ]:
all_results_json = {
    'timestamp': datetime.now().isoformat(),
    'phase': 3,
    'answer_verification': {},
    'nlp_metrics_answer_verification': nlp_df.to_dict(),
    'question_generation_nlp': {
        'template_only': qg_template_metrics,
        'ml_ranker':     qg_ranked_metrics,
    }
}

for name, m in all_metrics.items():
    key = name.lower().replace(' ','_').replace('(','').replace(')','')
    safe = {k: v for k, v in m.items()}
    all_results_json['answer_verification'][key] = safe

with open(f'{MODEL_DIR}/phase3_all_results.json', 'w') as f:
    json.dump(all_results_json, f, indent=2, default=str)

results_df.to_csv(f'{MODEL_DIR}/phase3_comparison_table.csv')
nlp_df.to_csv(f'{MODEL_DIR}/phase3_nlp_metrics.csv')

print('Saved: phase3_all_results.json')
print('Saved: phase3_comparison_table.csv')
print('Saved: phase3_nlp_metrics.csv')
print()
print('Files in models/model_a/traditional/:')
for fp in sorted(Path(MODEL_DIR).glob('*')):
    print(f'  {fp.name:<50} {fp.stat().st_size/1024:>8.1f} KB')